# Notebook 3: Audio Input with Allosaurus

## What this notebook covers
- Running real audio through a phoneme recogniser (allosaurus), not typing phonemes by hand
- Allosaurus outputs raw IPA phones directly, no G2P needed on the hypothesis side
- Wiring real audio into the PronunciationEvaluator built in Notebook 1
- Where this breaks down (accents, mic quality, short names) and why that matters for a take-home

Built as preparation for the NameCoach take-home evaluation task.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
sys.path.append('../src')

from g2p import g2p
from per import phoneme_error_rate, error_breakdown
from evaluator import PronunciationEvaluator

In [3]:
# Libraries already installed via pip in terminal
from allosaurus.app import read_recognizer

# This downloads/loads the pretrained model on first run, can take a moment
model = read_recognizer()

print("Allosaurus model loaded.")

Allosaurus model loaded.


In [4]:
import soundfile as sf
import os

audio_dir = "audio"

files = {
    "hello": "hello.mp3.mpeg",
    "slow_hello": "hello_slow.mp3.mpeg",
    "slow_hello2": "better_hello.mp3.mpeg",
    "arjun": "arjun.mp3.mpeg",
    "saoirse": "saoirse.mp3.mpeg",
}

wav_paths = {}

for name, fname in files.items():
    src = os.path.join(audio_dir, fname)
    dst = os.path.join(audio_dir, f"{name}.wav")
    try:
        data, samplerate = sf.read(src)
        sf.write(dst, data, samplerate)
        wav_paths[name] = dst
        print(f"{name}: OK, {samplerate} Hz, {len(data)/samplerate:.2f}s -> {dst}")
    except Exception as e:
        print(f"{name}: FAILED, {type(e).__name__}: {e}")

hello: OK, 48000 Hz, 3.10s -> audio\hello.wav
slow_hello: OK, 48000 Hz, 1.70s -> audio\slow_hello.wav
slow_hello2: OK, 48000 Hz, 3.34s -> audio\slow_hello2.wav
arjun: FAILED, LibsndfileError: Unspecified internal error.
saoirse: OK, 48000 Hz, 3.38s -> audio\saoirse.wav


In [5]:
import static_ffmpeg
static_ffmpeg.add_paths()  # fetches ffmpeg/ffprobe binaries on first run, then wires them into PATH

from pydub import AudioSegment

src = "audio/arjun.mp3.mpeg"
dst = "audio/arjun.wav"

# from_file() with no format= lets ffmpeg sniff the real container,
# instead of trusting the (possibly wrong) file extension
audio = AudioSegment.from_file(src)
audio.export(dst, format="wav")

print(f"Converted arjun: {len(audio)/1000:.2f}s, {audio.frame_rate} Hz")

Converted arjun: 2.83s, 48000 Hz


In [6]:
import soundfile as sf

for name in ["hello", "arjun", "saoirse", "slow_hello", "slow_hello2"]:
    path = f"audio/{name}.wav"
    data, sr = sf.read(path)
    print(f"{name}: {sr} Hz, {len(data)/sr:.2f}s, {len(data)} samples")

hello: 48000 Hz, 3.10s, 148608 samples
arjun: 48000 Hz, 2.83s, 135936 samples
saoirse: 48000 Hz, 3.38s, 162432 samples
slow_hello: 48000 Hz, 1.70s, 81792 samples
slow_hello2: 48000 Hz, 3.34s, 160128 samples


In [7]:
from allosaurus.app import read_recognizer

model = read_recognizer()

results = {}
for name in ["hello", "arjun", "saoirse", "slow_hello", "slow_hello2"]:
    path = f"audio/{name}.wav"
    hypothesis = model.recognize(path)
    results[name] = hypothesis.split()
    print(f"{name}: {results[name]}")

hello: ['a', 'l', 'o']
arjun: ['a', 'ɾ', 'd͡ʒ', 'y', 'ʊ', 'uː']
saoirse: ['s', 'œ', 'ɾ', 't͡ʃʲ', 'ɻ̩']
slow_hello: ['ɒ', 'l', 'ɔ', 'uə', 'ɪ']
slow_hello2: ['a', 'l', 'o']


In [8]:
evaluator = PronunciationEvaluator()

# Multiple takes of the same word need to map back to that word for
# reference lookup. "word" is never the recording's label, it's always
# the actual target word.
word_for = {
    "hello": "hello",
    "slow_hello": "hello",
    "slow_hello2": "hello",
    "arjun": "Arjun",
    "saoirse": "Saoirse",
}

for label in ["hello", "slow_hello", "slow_hello2", "arjun", "saoirse"]:
    result = evaluator.evaluate(word=word_for[label], spoken_phonemes=results[label])
    print(f"[{label}]")
    evaluator.print_report(result)

PronunciationEvaluator ready.
[hello]

Word:           hello
Reference:      ['h', 'ə', 'l', 'oʊ']
  Source:       CMUdict (confidence: HIGH)
Spoken:         ['a', 'l', 'o']
PER:            0.75
Accuracy:       25/100
Grade:          POOR
--------------------------------------------------
[slow_hello]

Word:           hello
Reference:      ['h', 'ə', 'l', 'oʊ']
  Source:       CMUdict (confidence: HIGH)
Spoken:         ['ɒ', 'l', 'ɔ', 'uə', 'ɪ']
PER:            1.25
Accuracy:       0/100
Grade:          POOR
--------------------------------------------------
[slow_hello2]

Word:           hello
Reference:      ['h', 'ə', 'l', 'oʊ']
  Source:       CMUdict (confidence: HIGH)
Spoken:         ['a', 'l', 'o']
PER:            0.75
Accuracy:       25/100
Grade:          POOR
--------------------------------------------------
[arjun]

Word:           Arjun
Reference:      ['ɑː', 'ɹ', 'd', 'ʒ', 'ʌ', 'n']
  Source:       phonemizer (confidence: LOW)
Spoken:         ['a', 'ɾ', 'd', 'ʒ', 'y', 'ʊ'

## Summary: What Notebook 3 taught us

### Real audio breaks assumptions Notebooks 1 and 2 got to skip
- CMUdict is ARPAbet, allosaurus is IPA. Comparing them directly means comparing two different alphabets, not two notations of the same one, an ARPAbet reference against a real audio hypothesis scores badly even for a perfect pronunciation
- Allosaurus fuses affricates into one tie-barred symbol (`d͡ʒ`), phonemizer and CMUdict both write them as two plain letters (`d`, `ʒ`). Same sound, different notation, same category of problem as Notebook 2's case bug: two systems disagreeing on how to write something down, not on what was actually said

### Two fixes, both verified against real espeak/allosaurus output before trusting them
- `arpabet_to_ipa()`: converts CMUdict's ARPAbet into IPA, including the AH0-vs-AH stress-conditioned schwa distinction, so CMUdict stays usable as high-confidence ground truth instead of getting replaced by synthetic phonemizer output
- `strip_tie_bars()`: splits allosaurus's fused affricates to match everyone else's convention. Applied to both sides of every comparison, symmetrically, the same lesson from Notebook 2's error_breakdown() bug applies here too

### Key numbers (after both fixes, real recorded audio)
| Word | Take | PER | Notes |
|---|---|---|---|
| hello | original | 0.75 | missing /h/, /oʊ/ flattened to /o/ |
| hello | larger/cleaner take | 0.75 | identical result, different recording |
| hello | short slow attempt | 1.25 | worse, not better, likely a clipped take |
| Arjun | single take | 0.833 | final /n/ dropped, vowels scattered |
| Saoirse | single take | 1.25 | first phoneme matched, rest diverged |

### The real finding
The repeated 0.75 across two separate "hello" recordings is the important result here, not the number itself. A one-off score could be recording noise, a repeatable score across independent takes is either a genuine feature of how this speaker realises the word (weak word-initial /h/, monophthongised diphthong, both normal in casual English) or a consistent allosaurus behaviour on this voice. Either way it's signal, not noise, and it wouldn't have shown up without recording more than one take.

The short "slow" attempt scoring worse than the natural-pace one is also worth keeping: over-enunciating for a recognizer's benefit can backfire, natural pace and duration matter as much as clarity of pronunciation.

This has a direct implication for how a real system should be built: **PER on real audio should probably never be expected to hit 0.0, even for correct pronunciation.** Universal phone recognisers carry their own systematic noise. A production scorer needs a calibrated "good enough" threshold established against known-correct baselines, and ideally multiple takes, rather than treating a single recognition pass as ground truth about what was actually said.

### What's next
- Notebook 4: full pipeline, text in, audio in, PER out, one clean function call
- Worth deciding whether Notebook 4 also tackles a calibration step (what PER range counts as "correct" given recognizer noise), given what this notebook found